In [47]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2025-04-14 08:59:35--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.3.33, 104.26.2.33, 172.67.70.149, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.3.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip’

book-crossings.zip  100%[===================>]  24.88M   125MB/s    in 0.2s    

2025-04-14 08:59:35 (125 MB/s) - ‘book-crossings.zip’ saved [26085508/26085508]

Archive:  book-crossings.zip
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


In [30]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})
df_books['title'] = df_books['title'].fillna('')
df_books['author'] = df_books['author'].fillna('')

df_books['text'] = df_books['title'] + " " + df_books['author']

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

# Step 3: Filter users with <200 ratings
user_counts = df_ratings['user'].value_counts()
active_users = user_counts[user_counts >= 200].index
df_ratings = df_ratings[df_ratings['user'].isin(active_users)]

# Step 4: Filter books with <100 ratings
book_counts = df_ratings['isbn'].value_counts()
popular_books = book_counts[book_counts >= 100].index
df_ratings = df_ratings[df_ratings['isbn'].isin(popular_books)]

# Optional: Filter df_books to match ratings
df_books = df_books[df_books['isbn'].isin(df_ratings['isbn'].unique())]

book_avg_ratings = df_ratings.groupby('isbn')['rating'].mean()

# Merge with recommendations
df_books = df_books.merge(book_avg_ratings, on='isbn', how='left')
df_books = df_books.rename(columns={'rating': 'avg_rating'})

,isbn,title,author,text,avg_rating
0,0440234743,The Testament,John Grisham,The Testament John Grisham,1.435484
1,0971880107,Wild Animus,Rich Shapero,Wild Animus Rich Shapero,0.435616
2,0446310786,To Kill a Mockingbird,Harper Lee,To Kill a Mockingbird Harper Lee,3.863309
3,0440225701,The Street Lawyer,JOHN GRISHAM,The Street Lawyer JOHN GRISHAM,1.746377
4,0804106304,The Joy Luck Club,Amy Tan,The Joy Luck Club Amy Tan,1.848837
...,...,...,...,...,...
94,0671001795,Two for the Dough,Janet Evanovich,Two for the Dough Janet Evanovich,2.182540
95,0312983271,Full House (Janet Evanovich's Full Series),Janet Evanovich,Full House (Janet Evanovich's Full Series) Jan...,2.126214
96,0440222656,The Horse Whisperer,Nicholas Evans,The Horse Whisperer Nicholas Evans,1.273224
97,0553280341,B Is for Burglar (Kinsey Millhone Mysteries (P...,Sue Grafton,B Is for Burglar (Kinsey Millhone Mysteries (P...,2.234375


In [48]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_books['text'])

# Reshape the avg_rating column into a sparse matrix with shape (n_books, 1)
ratings_matrix = csr_matrix(df_books['avg_rating'].values.reshape(-1, 1))

# Combine TF-IDF with the ratings column
hybrid_matrix = hstack([tfidf_matrix, ratings_matrix])

In [49]:
model = NearestNeighbors(n_neighbors=5, metric='cosine', algorithm='brute')

model.fit(hybrid_matrix)

NearestNeighbors(algorithm='brute', metric='cosine')

In [53]:
# function to return recommended books - this will be tested
def get_recommends(book = ""):
  book_idx = df_books[df_books['title'].str.lower() == book.lower()].index[0]

  distances, indices = model.kneighbors(hybrid_matrix[book_idx], n_neighbors=10)
  # Build recommendations list (skip the first one because it's the book itself)
  recs = []
  for i in range(1, len(distances[0])):  # start from 1 to skip the input book
      title = df_books.iloc[indices[0][i]]['title']
      similarity = 1 - distances[0][i]  # convert distance to similarity (cosine similarity = 1 - distance)
      recs.append((title, round(similarity, 3)))  # round to match test precision

  return (book, recs)


In [54]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

("Where the Heart Is (Oprah's Book Club (Paperback))", [('Harry Potter and the Order of the Phoenix (Book 5)', np.float64(0.909)), ("She's Come Undone (Oprah's Book Club (Paperback))", np.float64(0.902)), ('To Kill a Mockingbird', np.float64(0.896)), ("Harry Potter and the Sorcerer's Stone (Harry Potter (Paperback))", np.float64(0.892)), ('Harry Potter and the Chamber of Secrets (Book 2)', np.float64(0.891)), ('The Five People You Meet in Heaven', np.float64(0.89)), ('Fahrenheit 451', np.float64(0.886)), ("Tuesdays with Morrie: An Old Man, a Young Man, and Life's Greatest Lesson", np.float64(0.884)), ('The Lovely Bones: A Novel', np.float64(0.884))])
You haven't passed yet. Keep trying!
